In [1]:
import pandas as pd
import numpy as np

In [2]:
#CARGAMOS EL ARCHIVO EXCEL

archivo = "Datos_CLV_2026_Tec_v2.xlsx"

ventas = pd.read_excel(
    archivo, sheet_name="Ventas_CLV_2022_2026"
)

existencias = pd.read_excel(
    archivo, sheet_name="Existencias_04_09_2026"
)

compras = pd.read_excel(
    archivo, sheet_name="Histórico_Compras"
)

clasificacion = pd.read_excel(
    archivo, sheet_name="Clasificación_Articulos"
)

salidas = pd.read_excel(
    archivo, sheet_name="Salidas_Almacen"
)


In [3]:
# NORMALIZAMOS LA KEY (CAMPO) DE RELACIÓN

bases = [ventas, existencias, compras,
         clasificacion, salidas]

for df in bases:
    df["articulo_id"] = pd.to_numeric(
        df["articulo_id"], errors="coerce"
    ).astype("Int64")

In [ ]:
# REVISAMOS LAS RELACIONES

for nombre, df in zip(
    ["Ventas", "Existencias", "Compras",
     "Clasificacion", "Salidas"], bases
):
    print("\nBase:", nombre)
    print("Registros:", len(df))
    print("Artículos únicos:",
          df["articulo_id"].nunique())
    print("Llaves duplicadas:",
          df["articulo_id"].duplicated().sum())
    print("Llaves nulas:",
          df["articulo_id"].isna().sum())

In [5]:
# AGREGAMOS LA INFORMACIÓN POR ARTICULO_ID
# Ventas
ventas["total_neto"] = pd.to_numeric(
    ventas["total_neto"], errors="coerce"
)

resumen_ventas = ventas.groupby(
    "articulo_id"
).agg(
    numero_ventas=("folio", "nunique"),
    unidades_vendidas=("total_unidades", "sum"),
    ingresos_totales=("total_neto", "sum"),
    precio_promedio=("precio_unitario", "mean")
).reset_index()

# Compras
resumen_compras = compras.groupby(
    "articulo_id"
).agg(
    numero_compras=("folio", "nunique"),
    unidades_compradas=("unidades", "sum"),
    importe_compras=("precio_total_neto", "sum")
).reset_index()

# Salidas de almacén
resumen_salidas = salidas.groupby(
    "articulo_id"
).agg(
    total_salidas=("salidas", "sum"),
    movimientos_almacen=("fecha", "count")
).reset_index()

# Clasificaciones
resumen_clasificacion = clasificacion.groupby(
    "articulo_id"
).agg(
    categorias=(
        "categoria",
        lambda x: " | ".join(
            sorted(x.dropna().astype(str).unique())
        )
    ),
    numero_categorias=("categoria", "nunique")
).reset_index()


In [6]:
# CONSOLIDAMOS LAS BASES

# Existencias funciona como base inicial.
# Se conserva también información de artículos
# que aparecen en otras fuentes mediante outer.

base_final = existencias.merge(
    resumen_ventas,
    on="articulo_id",
    how="outer",
    validate="one_to_one"
)

base_final = base_final.merge(
    resumen_compras,
    on="articulo_id",
    how="outer",
    validate="one_to_one"
)

base_final = base_final.merge(
    resumen_salidas,
    on="articulo_id",
    how="outer",
    validate="one_to_one"
)

base_final = base_final.merge(
    resumen_clasificacion,
    on="articulo_id",
    how="outer",
    validate="one_to_one"
)

In [7]:
# COMPLETAMOS VALORES NUMÉRICOS

columnas_numericas = [
    "existencias",
    "valor_total_inventario",
    "numero_ventas",
    "unidades_vendidas",
    "ingresos_totales",
    "numero_compras",
    "unidades_compradas",
    "importe_compras",
    "total_salidas",
    "movimientos_almacen",
    "numero_categorias"
]

for col in columnas_numericas:
    if col in base_final.columns:
        base_final[col] = base_final[col].fillna(0)

In [8]:
# CREAMOS VARIABLES PARA EL ANÁLISIS

base_final["precio_promedio"] = (
    base_final["precio_promedio"].fillna(0)
)

base_final["rotacion_salida"] = np.where(
    base_final["existencias"] > 0,
    base_final["total_salidas"] /
    base_final["existencias"],
    np.nan
)

base_final["diferencia_compras_ventas"] = (
    base_final["unidades_compradas"] -
    base_final["unidades_vendidas"]
)

In [9]:
#RESULTADO FINAL

print("\nDIMENSIONES DE LA BASE CONSOLIDADA")
print(base_final.shape)

print("\nPRIMEROS REGISTROS")
print(base_final.head())

print("\nINFORMACIÓN GENERAL")
print(base_final.info())



DIMENSIONES DE LA BASE CONSOLIDADA
(6055, 19)

PRIMEROS REGISTROS
   articulo_id nombre_articulo marca nombre_almacen  existencias  \
0         3452             NaN   NaN            NaN          0.0   
1         3455             NaN   NaN            NaN          0.0   
2         3458             NaN   NaN            NaN          0.0   
3         3461             NaN   NaN            NaN          0.0   
4         3464             NaN   NaN            NaN          0.0   

   valor_total_inventario  numero_ventas  unidades_vendidas  ingresos_totales  \
0                     0.0            0.0                0.0              0.00   
1                     0.0            0.0                0.0              0.00   
2                     0.0            1.0                3.0           6067.24   
3                     0.0            0.0                0.0              0.00   
4                     0.0            0.0                0.0              0.00   

   precio_promedio  numero_compras  u

In [ ]:
# EXPORTAMOS LA BASE UNIFICADA


base_final.to_excel(
    "Base_Consolidada_Articulos_Versión_inicial.xlsx",
    index=False
)

print("\nBase consolidada generada correctamente")



Base consolidada generada correctamente
